# CUDA Case Study - Parallel RSA-2048 Encryption / Decryption
**Abdullah (1BY23CS002) - Parallel Computing (BCS702) CCA 1**

How to run:
1. **Runtime -> Change runtime type -> T4 GPU -> Save**
2. **Runtime -> Run all**

The notebook writes the project files, compiles them with `nvcc` for whatever GPU Colab gives you, runs the narrated demo (the 5 CUDA steps), the correctness suite, and an independent Python check. The full benchmark at the end is optional (about 10 minutes).

In [ ]:
# 1. Check the GPU and pick the matching CUDA architecture
import os, subprocess
os.environ['PATH'] += ':/usr/local/cuda/bin'
r = subprocess.run(['nvidia-smi', '--query-gpu=name,compute_cap', '--format=csv,noheader'],
                   capture_output=True, text=True)
if r.returncode != 0:
    raise SystemExit('No GPU found: use Runtime -> Change runtime type -> T4 GPU, then run again.')
name, cc = [x.strip() for x in r.stdout.strip().splitlines()[0].split(',')]
ARCH = 'sm_' + cc.replace('.', '')
print(f'GPU: {name}  (compute capability {cc} -> {ARCH})')
!nvcc --version | tail -2

In [ ]:
# 2. Project folder
os.makedirs('/content/CUDA_RSA_CCA/src', exist_ok=True)
os.makedirs('/content/CUDA_RSA_CCA/tools', exist_ok=True)
%cd /content/CUDA_RSA_CCA

## Project source files
`bigint.cuh` = 2048-bit Montgomery arithmetic (shared by CPU and GPU), `rsa_key.h` = demo RSA-2048 key, `rsa_cuda.cu` = kernels + 5-step host pipeline + demo/verify/bench modes.

In [ ]:
%%writefile src/bigint.cuh
// =============================================================================
//  bigint.cuh  -  Multi-precision (big integer) arithmetic for RSA
//
//  Every function here is  __host__ __device__ : the SAME source code is
//  compiled once for the CPU (sequential baseline) and once for the GPU
//  (CUDA kernels).  The CPU-vs-GPU comparison therefore measures the effect
//  of parallel hardware, not of two different algorithms.
//
//  Number representation
//    A big number is an array of NL 32-bit "limbs", little-endian
//    (limb 0 is the least significant).  RSA-2048 -> NL = 64 limbs.
//
//  Core algorithm: Montgomery multiplication (CIOS variant, Koc et al. 1996)
//    MontMul(a, b) = a * b * R^-1 mod n,   R = 2^(32*NL)
//    It replaces the expensive "divide by n" of a normal modular multiply by
//    cheap shifts, so modular exponentiation becomes ~3000 MontMuls.
//
//  Register-friendly convention used by mont_mul(r, a, b, ...):
//    * 'a' is only ever indexed with compile-time constants (fully unrolled
//      inner loops), so on the GPU it lives in REGISTERS.
//    * 'b' is indexed with the run-time outer-loop counter, so it lives in
//      (L1-cached) local memory.  Only NL reads of 'b' happen per MontMul.
// =============================================================================
#pragma once
#include <stdint.h>

#ifdef __CUDACC__
#define HD __host__ __device__ __forceinline__
#else
#define HD inline
#endif

// Loop-unrolling hints: honoured by nvcc for the device pass, ignored on host.
#if defined(__CUDA_ARCH__)
#define UNROLL_ALL _Pragma("unroll")
#define UNROLL_NONE _Pragma("unroll 1")
#else
#define UNROLL_ALL
#define UNROLL_NONE
#endif

// ---------------------------------------------------------------------------
//  Parameter blocks (plain structs so they can be placed in global, shared or
//  constant memory without any change to the arithmetic code).
// ---------------------------------------------------------------------------
template <int NL>
struct MontCtx {
    uint32_t n[NL];   // odd modulus
    uint32_t r2[NL];  // R^2 mod n   (converts a number into Montgomery form)
    uint32_t n0inv;   // -n^-1 mod 2^32
};

// One modular exponentiation "key": base^exp mod n
template <int NL>
struct ExpParams {
    MontCtx<NL> m;
    uint32_t exp[NL];
    int exp_bits;     // bit length of exp (index of highest set bit + 1)
};

// RSA private key in CRT form (Chinese Remainder Theorem), H = NL/2 limbs
template <int H>
struct CrtParams {
    ExpParams<H> P;      // modulus p, exponent dp = d mod (p-1)
    ExpParams<H> Q;      // modulus q, exponent dq = d mod (q-1)
    uint32_t qinv_r[H];  // (q^-1 mod p) * R mod p   (Montgomery form)
};

// ---------------------------------------------------------------------------
//  Small helpers
// ---------------------------------------------------------------------------
template <int NL>
HD void bn_copy(uint32_t* r, const uint32_t* a) {
    UNROLL_ALL
    for (int i = 0; i < NL; ++i) r[i] = a[i];
}

template <int NL>
HD void bn_set_word(uint32_t* r, uint32_t w) {
    UNROLL_ALL
    for (int i = 0; i < NL; ++i) r[i] = 0;
    r[0] = w;
}

// Branch-free: returns 1 if (hi:x) >= n else 0, where 'hi' is an extra top word.
template <int NL>
HD uint32_t bn_geq_with_top(const uint32_t* x, uint32_t hi, const uint32_t* n) {
    uint32_t br = 0;
    UNROLL_ALL
    for (int j = 0; j < NL; ++j) {
        const uint64_t s = (uint64_t)x[j] - n[j] - br;
        br = (uint32_t)(s >> 32) & 1u;          // 1 if the subtraction borrowed
    }
    return (hi >= br) ? 1u : 0u;                // (hi*2^(32NL) + x) - n >= 0 ?
}

// Branch-free: r = x - (n AND mask);  mask is 0 or 0xFFFFFFFF. Returns borrow.
template <int NL>
HD uint32_t bn_sub_masked(uint32_t* r, const uint32_t* x, const uint32_t* n, uint32_t mask) {
    uint32_t br = 0;
    UNROLL_ALL
    for (int j = 0; j < NL; ++j) {
        const uint64_t s = (uint64_t)x[j] - (n[j] & mask) - br;
        r[j] = (uint32_t)s;
        br = (uint32_t)(s >> 32) & 1u;
    }
    return br;
}

// If x >= n then x -= n  (requires x < 2n).  Constant-time, no branches.
template <int NL>
HD void bn_cond_sub(uint32_t* x, const uint32_t* n) {
    const uint32_t mask = 0u - bn_geq_with_top<NL>(x, 0u, n);
    bn_sub_masked<NL>(x, x, n, mask);
}

// r = (a + b) mod n   (requires a, b < n)
template <int NL>
HD void bn_mod_add(uint32_t* r, const uint32_t* a, const uint32_t* b, const uint32_t* n) {
    uint32_t s[NL];
    uint32_t c = 0;
    UNROLL_ALL
    for (int j = 0; j < NL; ++j) {
        const uint64_t t = (uint64_t)a[j] + b[j] + c;
        s[j] = (uint32_t)t;
        c = (uint32_t)(t >> 32);
    }
    const uint32_t mask = 0u - bn_geq_with_top<NL>(s, c, n);
    bn_sub_masked<NL>(r, s, n, mask);
}

// r = (a - b) mod n   (requires a, b < n)
template <int NL>
HD void bn_mod_sub(uint32_t* r, const uint32_t* a, const uint32_t* b, const uint32_t* n) {
    uint32_t br = 0;
    UNROLL_ALL
    for (int j = 0; j < NL; ++j) {
        const uint64_t s = (uint64_t)a[j] - b[j] - br;
        r[j] = (uint32_t)s;
        br = (uint32_t)(s >> 32) & 1u;
    }
    const uint32_t mask = 0u - br;              // negative result -> add n back
    uint32_t c = 0;
    UNROLL_ALL
    for (int j = 0; j < NL; ++j) {
        const uint64_t s = (uint64_t)r[j] + (n[j] & mask) + c;
        r[j] = (uint32_t)s;
        c = (uint32_t)(s >> 32);
    }
}

// ---------------------------------------------------------------------------
//  Montgomery multiplication, CIOS (Coarsely Integrated Operand Scanning)
//
//    r = a * b * R^-1 mod n        with R = 2^(32*NL)
//
//  Pre-condition : a * b < R * n   (true if a,b < n, or if a < R and b < n)
//  Post-condition: 0 <= r < n      (fully reduced)
//  r may alias a (the result is written only at the very end).
// ---------------------------------------------------------------------------
template <int NL, class Ctx>
HD void mont_mul(uint32_t* r, const uint32_t* a, const uint32_t* b, const Ctx& ctx) {
    uint32_t t[NL + 2];
    UNROLL_ALL
    for (int j = 0; j < NL + 2; ++j) t[j] = 0;

    UNROLL_NONE
    for (int i = 0; i < NL; ++i) {
        // (1) t = t + a * b[i]
        const uint32_t bi = b[i];
        uint64_t C = 0;
        UNROLL_ALL
        for (int j = 0; j < NL; ++j) {
            const uint64_t s = (uint64_t)a[j] * bi + t[j] + C;   // max = 2^64 - 1, no overflow
            t[j] = (uint32_t)s;
            C = s >> 32;
        }
        uint64_t s = (uint64_t)t[NL] + C;
        t[NL] = (uint32_t)s;
        t[NL + 1] = (uint32_t)(s >> 32);

        // (2) m chosen so that t + m*n is divisible by 2^32;  then t = (t + m*n) / 2^32
        const uint32_t m = t[0] * ctx.n0inv;
        s = (uint64_t)m * ctx.n[0] + t[0];                      // low word is 0 by construction
        C = s >> 32;
        UNROLL_ALL
        for (int j = 1; j < NL; ++j) {
            s = (uint64_t)m * ctx.n[j] + t[j] + C;
            t[j - 1] = (uint32_t)s;
            C = s >> 32;
        }
        s = (uint64_t)t[NL] + C;
        t[NL - 1] = (uint32_t)s;
        t[NL] = t[NL + 1] + (uint32_t)(s >> 32);
    }

    // (3) t < 2n here: one conditional subtraction, done without branches
    const uint32_t mask = 0u - bn_geq_with_top<NL>(t, t[NL], ctx.n);
    bn_sub_masked<NL>(r, t, ctx.n, mask);
}

// ---------------------------------------------------------------------------
//  Modular exponentiation:  out = base ^ exp mod n
//  Left-to-right binary "square-and-multiply" in the Montgomery domain.
//  base may be any value < R (it is reduced by the first MontMul).
//
//  Because every connection on the server is decrypted with the SAME private
//  exponent d, all 32 threads of a warp take the same "if (bit)" decision at
//  the same time -> zero branch divergence on the GPU.
// ---------------------------------------------------------------------------
template <int NL, class P>
HD void mod_exp(uint32_t* out, const uint32_t* base, const P& prm) {
    uint32_t acc[NL];   // accumulator            (registers on the GPU)
    uint32_t xm[NL];    // base in Montgomery form (read as 'b' -> local memory)
    uint32_t tmp[NL];   // copy of acc used as 'b' operand when squaring

    if (prm.exp_bits <= 0) {             // x^0 = 1
        bn_set_word<NL>(out, 1u);
        return;
    }
    bn_copy<NL>(acc, base);
    mont_mul<NL>(xm, acc, prm.m.r2, prm.m);   // xm = base * R mod n
    bn_copy<NL>(acc, xm);                     // top bit of exp is always 1

    UNROLL_NONE
    for (int i = prm.exp_bits - 2; i >= 0; --i) {
        bn_copy<NL>(tmp, acc);
        mont_mul<NL>(acc, acc, tmp, prm.m);                       // square
        if ((prm.exp[i >> 5] >> (i & 31)) & 1u)
            mont_mul<NL>(acc, acc, xm, prm.m);                    // multiply
    }

    bn_set_word<NL>(tmp, 1u);
    mont_mul<NL>(out, acc, tmp, prm.m);       // leave Montgomery domain: acc * R^-1
}

// ---------------------------------------------------------------------------
//  x mod p for a 2H-limb x and an H-limb modulus p whose top bit is set.
//    x = xh * R + xl   ->   x mod p = (MontMul(xh, R^2) + (xl mod p)) mod p
// ---------------------------------------------------------------------------
template <int H>
HD void bn_reduce_double(uint32_t* r, const uint32_t* x, const MontCtx<H>& ctx) {
    uint32_t hi[H], lo[H];
    bn_copy<H>(hi, x + H);
    mont_mul<H>(hi, hi, ctx.r2, ctx);    // xh*R mod p   (xh < R, r2 < p: pre-condition holds)
    bn_copy<H>(lo, x);
    bn_cond_sub<H>(lo, ctx.n);           // xl < 2^(32H) <= 2p  ->  one subtraction suffices
    bn_mod_add<H>(r, hi, lo, ctx.n);
}

// out[2H] = h[H] * q[H] + add[H]   (schoolbook multiplication)
// Both loops are fully unrolled on the GPU: every index is a compile-time
// constant, so 'out' stays in registers.  (A version with a rolled outer loop,
// i.e. run-time indexed read-modify-write of out[i+j] in local memory, was
// compiled INCORRECTLY by NVCC 11.7 when inlined into the CRT kernel - see
// the report, section "Challenges".  The verification suite caught it.)
template <int H>
HD void bn_mul_add(uint32_t* out, const uint32_t* h, const uint32_t* q, const uint32_t* add) {
    UNROLL_ALL
    for (int k = 0; k < 2 * H; ++k) out[k] = (k < H) ? add[k] : 0u;
    UNROLL_ALL
    for (int i = 0; i < H; ++i) {
        const uint32_t hi = h[i];
        uint64_t C = 0;
        UNROLL_ALL
        for (int j = 0; j < H; ++j) {
            const uint64_t s = (uint64_t)hi * q[j] + out[i + j] + C;
            out[i + j] = (uint32_t)s;
            C = s >> 32;
        }
        out[i + H] = (uint32_t)C;        // this word is still 0 before row i
    }
}

// ---------------------------------------------------------------------------
//  RSA decryption with the Chinese Remainder Theorem (what real servers do):
//    m1 = c^dp mod p,  m2 = c^dq mod q,  h = qinv*(m1-m2) mod p,  m = m2 + h*q
//  Two half-size exponentiations ~ 3-4x less work than one full-size one.
// ---------------------------------------------------------------------------
template <int NL, class CP>
HD void rsa_crt_decrypt(uint32_t* m, const uint32_t* c, const CP& k) {
    constexpr int H = NL / 2;
    uint32_t cp[H], cq[H], m1[H], m2[H], h[H];

    bn_reduce_double<H>(cp, c, k.P.m);
    bn_reduce_double<H>(cq, c, k.Q.m);
    mod_exp<H>(m1, cp, k.P);
    mod_exp<H>(m2, cq, k.Q);
    bn_mod_sub<H>(h, m1, m2, k.P.m.n);        // m2 < q < p, so m2 is a valid residue mod p
    mont_mul<H>(h, h, k.qinv_r, k.P.m);       // h = (m1-m2) * qinv mod p
    bn_mul_add<H>(m, h, k.Q.m.n, m2);         // m = h*q + m2   (< n)
}


In [ ]:
%%writefile src/rsa_key.h
// AUTO-GENERATED by tools/keygen.py - do not edit by hand.
// RSA-2048 DEMONSTRATION key (deterministic seed "1BY23CS002-RSA-CUDA-CCA").
// All big numbers are little-endian arrays of 32-bit limbs (limb 0 = least significant).
#pragma once
#include <stdint.h>

#define RSA_KEY_BITS 2048
#define RSA_NLIMBS 64
#define RSA_HLIMBS 32
#define RSA_E 65537u
#define RSA_E_BITS 17
#define RSA_D_BITS 2046
#define RSA_DP_BITS 1023
#define RSA_DQ_BITS 1022
#define RSA_N0INV 0x21514f7fu
#define RSA_P0INV 0x76a6988bu
#define RSA_Q0INV 0x98b4b0a3u

// ---- public key (n, e) and private exponent d ----
static const uint32_t RSA_N[64] = {
    0x14e18f81u, 0x5cc4c91du, 0x9d7b786fu, 0x282f0d27u, 0xf6ab53d8u, 0x40cdd230u,
    0xcf9c625du, 0x9cbbc2e2u, 0x339da1e2u, 0xad5493d3u, 0x7f022c67u, 0x17787221u,
    0xf282546bu, 0x8beed16au, 0x481c9ec1u, 0x015ba18au, 0xa800c036u, 0x9c6a6cf1u,
    0x902c209eu, 0xd188be30u, 0xc39aeef1u, 0x43173531u, 0x1d97c17au, 0x37614f8cu,
    0xde83663au, 0x2e78f228u, 0x3215357fu, 0xfe4be6e8u, 0x174037e8u, 0x4eb180b8u,
    0xa45d3a8bu, 0xc0f02d38u, 0x078b938du, 0x8606d8b8u, 0x07244d4eu, 0xde3216c5u,
    0x4ed1203au, 0x1826e297u, 0x42c7c625u, 0x703b81a3u, 0x558af149u, 0x4136b29cu,
    0x9632a4d3u, 0x1d40d009u, 0xdbbe43d6u, 0xae09e1a4u, 0x5c25a7c5u, 0x03285232u,
    0x9f175325u, 0xf3ccc773u, 0x97094b8au, 0x1e19af7eu, 0x86596b99u, 0x38a122b6u,
    0x3004e797u, 0x0587159fu, 0xf0eec8aeu, 0xcaaf40b1u, 0xa0824c0au, 0x9492f8dau,
    0xefbdae83u, 0xccdb4545u, 0x4e5b8566u, 0xa8289537u
};
static const uint32_t RSA_D[64] = {
    0x92316835u, 0x8fac5d83u, 0x0307cc76u, 0x1cecd6f5u, 0xf8aefc96u, 0x5c9fd381u,
    0x5f14026au, 0xa4ef552du, 0x6e2de4a5u, 0xf262ddf0u, 0xf193b754u, 0xf54ef507u,
    0xce75244fu, 0x2ae96842u, 0x4c51c181u, 0xa16f565fu, 0x2f34e5e3u, 0x92b84805u,
    0x0e699ef8u, 0x28f715f5u, 0xaba2b1ceu, 0xd179a9a8u, 0x394f23eau, 0x2578a6ceu,
    0x1ce69d7fu, 0x085a3d7fu, 0x30cb8e78u, 0x41d5fdeeu, 0xfa431603u, 0xc65890f8u,
    0x0be96928u, 0x4e97af2bu, 0x2bf3514du, 0x326d1004u, 0x32e35c46u, 0x9835f821u,
    0xbbc13b6cu, 0xa8850b36u, 0xf81f2379u, 0x167b2b76u, 0xb5c075f0u, 0x5def8b55u,
    0x86030a26u, 0xf0aba930u, 0x18fb2b56u, 0xc3150875u, 0x53ec44b8u, 0x6a2c05fbu,
    0xdf96202au, 0xb406e64cu, 0x9116ceb0u, 0xe89c7842u, 0x4cb14b5cu, 0x9db5f7e9u,
    0x823b5959u, 0xd6e6b973u, 0x554e6f3au, 0x4e53b577u, 0xa7afcaa2u, 0x3ddd8f97u,
    0x9e0d19ecu, 0x47b2752bu, 0x254b21a9u, 0x234ce7acu
};
static const uint32_t RSA_R2N[64] = {
    0xc9d4493bu, 0x369567c6u, 0x8928f457u, 0x20b26595u, 0x3d666616u, 0x97b48f29u,
    0x1a7a279cu, 0x5ec1f5ddu, 0x71f4e134u, 0x6f66950au, 0x1bc4671du, 0x9700ad77u,
    0x1c9ddf23u, 0xf9ec1dd6u, 0xfd4de32cu, 0xfac8ed33u, 0x24d482edu, 0x0eb28e8eu,
    0x2a38c9eau, 0x5248cd3bu, 0x7393a743u, 0xdcdd6fa5u, 0x7df1038au, 0xc6838e62u,
    0xe57908a9u, 0x5df04282u, 0x64bd2c60u, 0xb3904ed4u, 0x955bbd0au, 0xcd859ea7u,
    0x7676ddc0u, 0xa10abf7bu, 0xbe3ab640u, 0x90f92148u, 0x807d6181u, 0x98d896b2u,
    0x8f39ef02u, 0xdca6a1f5u, 0x23460effu, 0x0a696704u, 0x0f1b570du, 0x5bdf2502u,
    0x3a3e362eu, 0xc76b6722u, 0x88c99544u, 0x5bd2c81bu, 0x1b5cb675u, 0x29e2f92fu,
    0x92fba51au, 0x019a8468u, 0x144160e1u, 0xe7cffa60u, 0x69592552u, 0xf8a44625u,
    0x84cf63bbu, 0x90dd2de8u, 0x9accd489u, 0x5c1fa242u, 0x85da5f47u, 0x11858952u,
    0x48e3a144u, 0xb76844fbu, 0x511150a1u, 0x0bd31027u
};

// ---- CRT private-key components (p > q) ----
static const uint32_t RSA_P[32] = {
    0x2304f0ddu, 0x6fffe387u, 0xc4bb4158u, 0xb122babbu, 0xc633bb9fu, 0xcc07f928u,
    0x558dede1u, 0x6b0ce2f7u, 0x0ddcae35u, 0x6cc23e6du, 0x9d494cb2u, 0xf3558c6du,
    0x59f14853u, 0xaae0e594u, 0xf2ce4793u, 0x3becd16au, 0xe445d71au, 0x92cfe449u,
    0x256ced1fu, 0x69d93472u, 0x70f02cb1u, 0x07189c61u, 0xc97a45dcu, 0x5c435769u,
    0x364f9debu, 0x0401932du, 0x84eb412cu, 0x8f969275u, 0x66c0584du, 0xb38923feu,
    0xe12a37c8u, 0xe00dbed6u
};
static const uint32_t RSA_Q[32] = {
    0x38ac7cf5u, 0x82aef210u, 0x7ee11395u, 0xf3e35607u, 0xcace0818u, 0xcb0ed0bdu,
    0xe0c4bc70u, 0xddc0e748u, 0x49d9ed36u, 0x74897d27u, 0x8de0a8a7u, 0x60bd0e42u,
    0x63dedd3eu, 0x282bed76u, 0xea8d7c00u, 0xabaf603eu, 0xb71a6fa0u, 0x50de8205u,
    0x4d5ddf02u, 0x6fe44ad5u, 0x98e8afb1u, 0xda492ef1u, 0xcbda8273u, 0x16262a5bu,
    0x45f1db78u, 0x3d5ddba8u, 0xb4f7183bu, 0xea116b46u, 0x2a62247eu, 0xbd4e44fbu,
    0x7fb48e64u, 0xc022971eu
};
static const uint32_t RSA_DP[32] = {
    0xe0a06ca9u, 0x24846f57u, 0xb852364cu, 0x2258f250u, 0xbcc9b5c7u, 0x129d9ba7u,
    0xc09460c6u, 0x20fd3f8fu, 0x8b697985u, 0xb1bc27adu, 0xe57aa90eu, 0xecba9ec2u,
    0x87458e56u, 0x6816b70bu, 0xc7821796u, 0xe291d7bfu, 0x681d36d6u, 0x5e02f10cu,
    0xa8f2462fu, 0x77a3b9eeu, 0x51eab4c3u, 0x269a61a2u, 0x10d0e714u, 0x6a8af3acu,
    0xb9613cadu, 0xe4715f74u, 0x5fdf85cbu, 0x058b672au, 0x0a0b1c1bu, 0x6b369f78u,
    0x14ff5105u, 0x5f86bcc5u
};
static const uint32_t RSA_DQ[32] = {
    0xe5c93439u, 0x5d8e33b0u, 0x0d12be29u, 0x6bba2ed2u, 0xb08da79cu, 0xabbd5d3au,
    0x755854a9u, 0x7c6f1b5fu, 0x15a87105u, 0xfc412a99u, 0x2eecc15cu, 0x5a2eeeaau,
    0xae52abf8u, 0xda24b0c1u, 0xdd7e95b9u, 0x072cb5a7u, 0x6b84ec35u, 0x7754ff17u,
    0xbaaf16ceu, 0x286a0a16u, 0x911393c7u, 0x1b6196a6u, 0x0d76e273u, 0x93f629fau,
    0x2a937430u, 0x22c54e10u, 0x17670802u, 0x8119a470u, 0xc23db374u, 0xd9c58951u,
    0x4274f627u, 0x25a32103u
};
static const uint32_t RSA_R2P[32] = {
    0x0495b343u, 0x933a2f3au, 0x93913d61u, 0xc1d4c50au, 0xe87d2c2au, 0x686c220du,
    0x23d67f2fu, 0x7cf940fbu, 0x61fb5eebu, 0xb903a371u, 0x0c1b9b25u, 0xea37cb98u,
    0x7a8f80a5u, 0x4c8f386fu, 0x01205b56u, 0xb04600a1u, 0x20e6eb1du, 0x67b7db63u,
    0xea4054ebu, 0xe2f743f4u, 0x8bd2caddu, 0xc8937276u, 0x0ea2e3a3u, 0x177b532eu,
    0x92d3efebu, 0x42143187u, 0x387e6a96u, 0x3c5e19bdu, 0xb0197730u, 0xa74de1f7u,
    0x25e4c4afu, 0xdafa74a8u
};
static const uint32_t RSA_R2Q[32] = {
    0xb045146bu, 0x433542adu, 0xf705f12eu, 0x59a083adu, 0x60962caeu, 0x216774ddu,
    0x5c6137e5u, 0x544b96e7u, 0xc7ecdae6u, 0x3910a7bdu, 0xd51f060cu, 0x7b663943u,
    0x2bb0b11du, 0xd7704a2bu, 0xa9786ba6u, 0xde6a22a3u, 0xd732a5b7u, 0xe9da9b20u,
    0x763774b2u, 0xbc6c4206u, 0x74acd82du, 0x15ff3820u, 0xb5016b69u, 0xb6ec8cd8u,
    0xb1c5505du, 0xb0126cf1u, 0xdc1014f9u, 0x91b7782du, 0xddf0a781u, 0x8bda35c7u,
    0x19f0728eu, 0xa6450fd1u
};
static const uint32_t RSA_QINV_R[32] = {
    0xed7bc7beu, 0x97a4f647u, 0xcb9f1a15u, 0xf65777c2u, 0xf95f18b8u, 0x3547d986u,
    0x64cd75feu, 0xdde36b70u, 0x5bfbdc24u, 0x6ead8153u, 0x018e01bdu, 0xefafe58eu,
    0x9346b487u, 0x2cd545d3u, 0x7ecbae00u, 0xa5f7292du, 0x4a400b9eu, 0xae22e08du,
    0x560008c2u, 0x69989fc7u, 0xb7367575u, 0x286fcf24u, 0x4c8dce09u, 0x6ab658ceu,
    0xe303f904u, 0x86fcefc4u, 0x6088a708u, 0xea380df0u, 0x19c1bd57u, 0x22dc8f54u,
    0x46ec7b2fu, 0xcb8b5f4du
};


In [ ]:
%%writefile src/rsa_cuda.cu
// =============================================================================
//  rsa_cuda.cu  -  CUDA Case Study: Cryptography (RSA Encryption / Decryption)
//
//  Scenario: a secure web server must decrypt thousands of RSA-2048 session
//  keys (one per incoming user connection) at the same time.  Every
//  decryption is an independent modular exponentiation  m = c^d mod n
//  -> "embarrassingly parallel":  ONE CUDA THREAD = ONE USER CONNECTION.
//
//  Modes
//    ./rsa_cuda demo   [N]   narrated end-to-end demo with readable messages
//    ./rsa_cuda verify [N]   correctness test suite (GPU vs CPU vs original)
//    ./rsa_cuda bench        performance experiments -> CSV files in results/
//    ./rsa_cuda info         GPU + kernel resource information
//    ./rsa_cuda all          demo + verify + bench   (default)
//  Options:  --out <dir>  (default "results")
// =============================================================================
#include <algorithm>
#include <chrono>
#include <cstdint>
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <filesystem>
#include <string>
#include <vector>

#include <cuda_runtime.h>
#ifdef _OPENMP
#include <omp.h>
#endif

#include "rsa_key.h"
#include "bigint.cuh"

constexpr int NL = RSA_NLIMBS;   // 64 limbs x 32 bit = 2048-bit numbers
constexpr int HL = RSA_HLIMBS;   // 32 limbs = 1024-bit CRT halves
static_assert(NL == 2 * HL, "CRT needs n = p*q with equal-size primes");

using Key    = ExpParams<NL>;
using CrtKey = CrtParams<HL>;

#define CUDA_CHECK(call)                                                          \
    do {                                                                          \
        cudaError_t err_ = (call);                                                \
        if (err_ != cudaSuccess) {                                                \
            fprintf(stderr, "CUDA error '%s' at %s:%d\n  -> %s\n", #call,         \
                    __FILE__, __LINE__, cudaGetErrorString(err_));                \
            exit(EXIT_FAILURE);                                                   \
        }                                                                         \
    } while (0)

// -----------------------------------------------------------------------------
//  Keys (host copies)
// -----------------------------------------------------------------------------
static Key    g_pub;    // (n, e)  public key  -> encryption
static Key    g_priv;   // (n, d)  private key -> decryption
static CrtKey g_crt;    // (p, q, dp, dq, qinv) -> CRT decryption

// Constant-memory copies used by the GPU kernels (64 KB constant cache,
// broadcast to all threads of a warp that read the same address).
__constant__ Key    c_key;
__constant__ CrtKey c_crt;

static int bit_length(const uint32_t* x, int nl) {
    for (int i = nl - 1; i >= 0; --i)
        for (int b = 31; b >= 0; --b)
            if ((x[i] >> b) & 1u) return 32 * i + b + 1;
    return 0;
}

static void build_keys() {
    auto fill_ctx = [](auto& ctx, const uint32_t* n, const uint32_t* r2, uint32_t n0inv, int nl) {
        memcpy(ctx.n, n, nl * 4);
        memcpy(ctx.r2, r2, nl * 4);
        ctx.n0inv = n0inv;
        // sanity: n * n0inv == -1 (mod 2^32)
        if ((uint32_t)(n[0] * n0inv) != 0xFFFFFFFFu) {
            fprintf(stderr, "Key error: inconsistent Montgomery constant\n");
            exit(EXIT_FAILURE);
        }
    };
    fill_ctx(g_pub.m, RSA_N, RSA_R2N, RSA_N0INV, NL);
    memset(g_pub.exp, 0, sizeof(g_pub.exp));
    g_pub.exp[0] = RSA_E;
    g_pub.exp_bits = RSA_E_BITS;

    g_priv.m = g_pub.m;
    memcpy(g_priv.exp, RSA_D, sizeof(g_priv.exp));
    g_priv.exp_bits = RSA_D_BITS;

    fill_ctx(g_crt.P.m, RSA_P, RSA_R2P, RSA_P0INV, HL);
    memcpy(g_crt.P.exp, RSA_DP, sizeof(g_crt.P.exp));
    g_crt.P.exp_bits = RSA_DP_BITS;
    fill_ctx(g_crt.Q.m, RSA_Q, RSA_R2Q, RSA_Q0INV, HL);
    memcpy(g_crt.Q.exp, RSA_DQ, sizeof(g_crt.Q.exp));
    g_crt.Q.exp_bits = RSA_DQ_BITS;
    memcpy(g_crt.qinv_r, RSA_QINV_R, sizeof(g_crt.qinv_r));

    if (bit_length(g_pub.exp, NL) != g_pub.exp_bits || bit_length(g_priv.exp, NL) != g_priv.exp_bits ||
        bit_length(g_crt.P.exp, HL) != g_crt.P.exp_bits || bit_length(g_crt.Q.exp, HL) != g_crt.Q.exp_bits ||
        bit_length(RSA_N, NL) != RSA_KEY_BITS) {
        fprintf(stderr, "Key error: bit lengths in rsa_key.h are inconsistent\n");
        exit(EXIT_FAILURE);
    }
}

// =============================================================================
//  GPU KERNELS
//
//  Data layout ("structure of arrays" / interleaved):
//      limb l of message i is stored at  buf[l * stride + i]
//  so when thread i and thread i+1 read limb l they touch neighbouring
//  32-bit words -> every warp-wide load/store is fully COALESCED.
// =============================================================================
enum class KeyMem { Global, Shared, Constant };
enum class Op { Encrypt, Decrypt, DecryptCRT };

static const char* mem_name(KeyMem m) {
    return m == KeyMem::Global ? "global" : m == KeyMem::Shared ? "shared" : "constant";
}
static const char* op_name(Op o) {
    return o == Op::Encrypt ? "encrypt" : o == Op::Decrypt ? "decrypt" : "decrypt-CRT";
}

// Work done by ONE thread = ONE user connection
template <class K>
__device__ __forceinline__ void modexp_one_connection(uint32_t* __restrict__ out,
                                                      const uint32_t* __restrict__ in,
                                                      int idx, int stride, const K& key) {
    uint32_t x[NL], y[NL];
#pragma unroll
    for (int l = 0; l < NL; ++l) x[l] = in[(size_t)l * stride + idx];    // coalesced load
    mod_exp<NL>(y, x, key);                                              // y = x^exp mod n
#pragma unroll
    for (int l = 0; l < NL; ++l) out[(size_t)l * stride + idx] = y[l];   // coalesced store
}

// Plain modular-exponentiation kernel (encryption with e, or decryption with d).
// The template parameter selects WHERE the key (modulus, R^2, exponent) is read
// from, so the three levels of the memory hierarchy can be compared directly.
template <KeyMem MEM>
__global__ void rsa_modexp_kernel(uint32_t* __restrict__ out, const uint32_t* __restrict__ in,
                                  int first, int count, int stride, const Key* __restrict__ gkey) {
    const int local = blockIdx.x * blockDim.x + threadIdx.x;

    if constexpr (MEM == KeyMem::Shared) {
        // The whole block cooperatively stages the key in on-chip shared memory once.
        __shared__ Key s_key;
        const uint32_t* src = reinterpret_cast<const uint32_t*>(gkey);
        uint32_t* dst = reinterpret_cast<uint32_t*>(&s_key);
        for (int w = threadIdx.x; w < (int)(sizeof(Key) / 4); w += blockDim.x) dst[w] = src[w];
        __syncthreads();                                   // before any early exit!
        if (local < count) modexp_one_connection(out, in, first + local, stride, s_key);
    } else if constexpr (MEM == KeyMem::Global) {
        if (local < count) modexp_one_connection(out, in, first + local, stride, *gkey);
    } else {
        if (local < count) modexp_one_connection(out, in, first + local, stride, c_key);
    }
}

// CRT decryption kernel (key in constant memory) - what production servers use.
__global__ void rsa_crt_kernel(uint32_t* __restrict__ out, const uint32_t* __restrict__ in,
                               int first, int count, int stride) {
    const int local = blockIdx.x * blockDim.x + threadIdx.x;
    if (local >= count) return;                            // boundary check
    const int idx = first + local;

    uint32_t c[NL], m[NL];
#pragma unroll
    for (int l = 0; l < NL; ++l) c[l] = in[(size_t)l * stride + idx];
    rsa_crt_decrypt<NL>(m, c, c_crt);
#pragma unroll
    for (int l = 0; l < NL; ++l) out[(size_t)l * stride + idx] = m[l];
}

// =============================================================================
//  HOST UTILITIES
// =============================================================================
using Clock = std::chrono::steady_clock;
static double ms_since(Clock::time_point t0) {
    return std::chrono::duration<double, std::milli>(Clock::now() - t0).count();
}

static std::string g_outdir = "results";
// Threads per kernel launch. On Windows/WSL (WDDM) the driver resets the GPU if
// one kernel runs longer than ~2 s (TDR watchdog), so a large batch is split
// into launches of ONE FULL WAVE each: exactly as many threads as the GPU can
// keep resident at once (from the occupancy calculator). 0 = automatic.
static int g_chunk = 0;
static int g_block = 64;

struct Rng {   // splitmix64 - small, fast, reproducible
    uint64_t s;
    explicit Rng(uint64_t seed) : s(seed) {}
    uint64_t next() {
        uint64_t z = (s += 0x9E3779B97F4A7C15ull);
        z = (z ^ (z >> 30)) * 0xBF58476D1CE4E5B9ull;
        z = (z ^ (z >> 27)) * 0x94D049BB133111EBull;
        return z ^ (z >> 31);
    }
    uint32_t u32() { return (uint32_t)(next() >> 32); }
};

// A batch of 'count' big numbers in interleaved layout
struct Batch {
    int count = 0;
    std::vector<uint32_t> w;
    explicit Batch(int c = 0) : count(c), w((size_t)NL * c, 0u) {}
    void get(int i, uint32_t* x) const { for (int l = 0; l < NL; ++l) x[l] = w[(size_t)l * count + i]; }
    void set(int i, const uint32_t* x) { for (int l = 0; l < NL; ++l) w[(size_t)l * count + i] = x[l]; }
    uint32_t* data() { return w.data(); }
    const uint32_t* data() const { return w.data(); }
};

// Uniformly random message m < n  (top limb strictly below n's top limb)
static void random_message(uint32_t* m, Rng& r) {
    for (int l = 0; l < NL; ++l) m[l] = r.u32();
    m[NL - 1] = r.u32() % RSA_N[NL - 1];
}

static Batch random_batch(int count, uint64_t seed) {
    Batch b(count);
    Rng r(seed);
    uint32_t m[NL];
    for (int i = 0; i < count; ++i) { random_message(m, r); b.set(i, m); }
    return b;
}

// text <-> big-endian integer (like PKCS#1 OS2IP / I2OSP, without padding)
static void text_to_bn(const std::string& s, uint32_t* x) {
    memset(x, 0, NL * 4);
    const int L = (int)s.size();
    for (int k = 0; k < L; ++k) {
        const int pos = L - 1 - k;                           // byte position from the LSB
        x[pos / 4] |= (uint32_t)(uint8_t)s[k] << (8 * (pos % 4));
    }
}
static std::string bn_to_text(const uint32_t* x) {
    std::string s;
    bool started = false;
    for (int pos = NL * 4 - 1; pos >= 0; --pos) {
        const uint8_t b = (uint8_t)(x[pos / 4] >> (8 * (pos % 4)));
        if (!started && b == 0) continue;
        started = true;
        s.push_back((b >= 32 && b < 127) ? (char)b : '?');
    }
    return s;
}
static std::string bn_hex(const uint32_t* x, int nl = NL) {
    std::string s;
    char buf[9];
    for (int i = nl - 1; i >= 0; --i) { snprintf(buf, sizeof buf, "%08x", x[i]); s += buf; }
    return s;
}

static int count_mismatches(const Batch& a, const Batch& b) {
    int bad = 0;
    uint32_t x[NL], y[NL];
    for (int i = 0; i < a.count; ++i) {
        a.get(i, x); b.get(i, y);
        if (memcmp(x, y, sizeof x) != 0) {
            if (getenv("RSA_DEBUG") && bad < 4) {
                int first_bad = -1;
                for (int l = 0; l < NL; ++l) if (x[l] != y[l]) { first_bad = l; break; }
                fprintf(stderr, "    mismatch at message %d (first differing limb %d)\n", i, first_bad);
            }
            ++bad;
        }
    }
    return bad;
}

static int max_cpu_threads() {
#ifdef _OPENMP
    return omp_get_max_threads();
#else
    return 1;
#endif
}

// =============================================================================
//  CPU BASELINE  (same bigint.cuh code, compiled for the host)
// =============================================================================
static double cpu_rsa_batch(Op op, const Batch& in, Batch& out, int threads) {
    const int count = in.count;
    const auto t0 = Clock::now();
#ifdef _OPENMP
#pragma omp parallel for num_threads(threads) schedule(dynamic, 1)
#endif
    for (int i = 0; i < count; ++i) {
        uint32_t x[NL], y[NL];
        in.get(i, x);
        if (op == Op::Encrypt)      mod_exp<NL>(y, x, g_pub);
        else if (op == Op::Decrypt) mod_exp<NL>(y, x, g_priv);
        else                        rsa_crt_decrypt<NL>(y, x, g_crt);
        out.set(i, y);
    }
    (void)threads;
    return ms_since(t0);
}

// =============================================================================
//  GPU PIPELINE - the standard 5 steps of every CUDA host program
// =============================================================================
struct GpuTimes {
    double alloc_ms = 0, h2d_ms = 0, kernel_ms = 0, d2h_ms = 0, free_ms = 0;
    double e2e() const { return h2d_ms + kernel_ms + d2h_ms; }   // data in -> results out
};

static void launch_rsa(Op op, KeyMem mem, uint32_t* d_out, const uint32_t* d_in,
                       int first, int n, int stride, const Key* d_key, int block) {
    const int grid = (n + block - 1) / block;   // ceil: may launch more threads than n
    if (op == Op::DecryptCRT) {
        rsa_crt_kernel<<<grid, block>>>(d_out, d_in, first, n, stride);
    } else if (mem == KeyMem::Global) {
        rsa_modexp_kernel<KeyMem::Global><<<grid, block>>>(d_out, d_in, first, n, stride, d_key);
    } else if (mem == KeyMem::Shared) {
        rsa_modexp_kernel<KeyMem::Shared><<<grid, block>>>(d_out, d_in, first, n, stride, d_key);
    } else {
        rsa_modexp_kernel<KeyMem::Constant><<<grid, block>>>(d_out, d_in, first, n, stride, d_key);
    }
    CUDA_CHECK(cudaGetLastError());
}

// Number of threads the whole GPU can hold at once for this kernel + block size
static int resident_threads(Op op, KeyMem mem, int block) {
    int blocksPerSM = 0;
    if (op == Op::DecryptCRT)
        CUDA_CHECK(cudaOccupancyMaxActiveBlocksPerMultiprocessor(&blocksPerSM, rsa_crt_kernel, block, 0));
    else if (mem == KeyMem::Global)
        CUDA_CHECK(cudaOccupancyMaxActiveBlocksPerMultiprocessor(&blocksPerSM, rsa_modexp_kernel<KeyMem::Global>, block, 0));
    else if (mem == KeyMem::Shared)
        CUDA_CHECK(cudaOccupancyMaxActiveBlocksPerMultiprocessor(&blocksPerSM, rsa_modexp_kernel<KeyMem::Shared>, block, 0));
    else
        CUDA_CHECK(cudaOccupancyMaxActiveBlocksPerMultiprocessor(&blocksPerSM, rsa_modexp_kernel<KeyMem::Constant>, block, 0));
    int dev = 0, sms = 0;
    CUDA_CHECK(cudaGetDevice(&dev));
    CUDA_CHECK(cudaDeviceGetAttribute(&sms, cudaDevAttrMultiProcessorCount, dev));
    return std::max(1, blocksPerSM) * block * sms;
}

static GpuTimes gpu_rsa_batch(Op op, KeyMem mem, const Batch& in, Batch& out, int block,
                              bool narrate = false) {
    GpuTimes t;
    const int count = in.count;
    const size_t bytes = (size_t)NL * count * sizeof(uint32_t);
    const Key& key = (op == Op::Encrypt) ? g_pub : g_priv;
    const int chunk = g_chunk > 0 ? g_chunk : resident_threads(op, mem, block);

    cudaEvent_t e0, e1, e2, e3;
    CUDA_CHECK(cudaEventCreate(&e0)); CUDA_CHECK(cudaEventCreate(&e1));
    CUDA_CHECK(cudaEventCreate(&e2)); CUDA_CHECK(cudaEventCreate(&e3));

    // ---- STEP 1: allocate device memory (host buffers already exist) --------
    auto c0 = Clock::now();
    uint32_t *d_in = nullptr, *d_out = nullptr;
    Key* d_key = nullptr;
    CUDA_CHECK(cudaMalloc(&d_in, bytes));
    CUDA_CHECK(cudaMalloc(&d_out, bytes));
    if (op != Op::DecryptCRT && mem != KeyMem::Constant) CUDA_CHECK(cudaMalloc(&d_key, sizeof(Key)));
    CUDA_CHECK(cudaDeviceSynchronize());
    t.alloc_ms = ms_since(c0);
    if (narrate)
        printf("  [Step 1] cudaMalloc   : d_in, d_out = %zu bytes each (%d x 2048-bit numbers)\n", bytes, count);

    // ---- STEP 2: copy inputs (and the key) Host -> Device --------------------
    CUDA_CHECK(cudaEventRecord(e0));
    CUDA_CHECK(cudaMemcpy(d_in, in.data(), bytes, cudaMemcpyHostToDevice));
    if (op == Op::DecryptCRT)          CUDA_CHECK(cudaMemcpyToSymbol(c_crt, &g_crt, sizeof(CrtKey)));
    else if (mem == KeyMem::Constant)  CUDA_CHECK(cudaMemcpyToSymbol(c_key, &key, sizeof(Key)));
    else                               CUDA_CHECK(cudaMemcpy(d_key, &key, sizeof(Key), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaEventRecord(e1));
    if (narrate)
        printf("  [Step 2] cudaMemcpy H->D: %d ciphertexts + key (%s memory)\n", count,
               op == Op::DecryptCRT ? "constant" : mem_name(mem));

    // ---- STEP 3: launch kernel  <<<grid, block>>>  (one thread per connection)
    for (int first = 0; first < count; first += chunk) {
        const int n = std::min(chunk, count - first);
        if (narrate)
            printf("  [Step 3] kernel %s<<<grid=%d, block=%d>>> [%s] -> %d threads for %d connections\n",
                   op == Op::DecryptCRT ? "rsa_crt_kernel" : "rsa_modexp_kernel",
                   (n + block - 1) / block, block, op_name(op), ((n + block - 1) / block) * block, n);
        launch_rsa(op, mem, d_out, d_in, first, n, count, d_key, block);
    }
    CUDA_CHECK(cudaEventRecord(e2));

    // ---- STEP 4: copy results Device -> Host ---------------------------------
    CUDA_CHECK(cudaMemcpy(out.data(), d_out, bytes, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaEventRecord(e3));
    CUDA_CHECK(cudaEventSynchronize(e3));
    if (narrate) printf("  [Step 4] cudaMemcpy D->H: %d results\n", count);

    float a, b, c;
    CUDA_CHECK(cudaEventElapsedTime(&a, e0, e1));
    CUDA_CHECK(cudaEventElapsedTime(&b, e1, e2));
    CUDA_CHECK(cudaEventElapsedTime(&c, e2, e3));
    t.h2d_ms = a; t.kernel_ms = b; t.d2h_ms = c;

    // ---- STEP 5: free device memory -----------------------------------------
    c0 = Clock::now();
    CUDA_CHECK(cudaFree(d_in));
    CUDA_CHECK(cudaFree(d_out));
    if (d_key) CUDA_CHECK(cudaFree(d_key));
    t.free_ms = ms_since(c0);
    if (narrate) printf("  [Step 5] cudaFree     : device buffers released\n");

    CUDA_CHECK(cudaEventDestroy(e0)); CUDA_CHECK(cudaEventDestroy(e1));
    CUDA_CHECK(cudaEventDestroy(e2)); CUDA_CHECK(cudaEventDestroy(e3));
    return t;
}

// Median of several GPU runs (the first CUDA call of a process pays for context
// creation, so benchmarks always perform a warm-up first).
static GpuTimes gpu_median(Op op, KeyMem mem, const Batch& in, Batch& out, int block, int reps) {
    std::vector<GpuTimes> v;
    for (int r = 0; r < reps; ++r) v.push_back(gpu_rsa_batch(op, mem, in, out, block));
    std::sort(v.begin(), v.end(), [](const GpuTimes& x, const GpuTimes& y) { return x.kernel_ms < y.kernel_ms; });
    return v[v.size() / 2];
}

static void gpu_warmup() {
    Batch in = random_batch(64, 7), out(64);
    gpu_rsa_batch(Op::Encrypt, KeyMem::Constant, in, out, g_block);
}

// =============================================================================
//  MODE: info
// =============================================================================
template <class F>
static void kernel_line(FILE* f, const char* name, F kernel, int block) {
    cudaFuncAttributes fa;
    CUDA_CHECK(cudaFuncGetAttributes(&fa, kernel));
    int blocksPerSM = 0;
    CUDA_CHECK(cudaOccupancyMaxActiveBlocksPerMultiprocessor(&blocksPerSM, kernel, block, 0));
    cudaDeviceProp p;
    CUDA_CHECK(cudaGetDeviceProperties(&p, 0));
    const int active = blocksPerSM * block;
    fprintf(f, "  %-34s regs/thread=%3d  local(spill)=%4zu B  smem=%4zu B  maxThr/blk=%4d  "
               "block=%d -> %d blocks/SM = %d threads/SM (occupancy %.1f%%), %d resident threads on GPU\n",
            name, fa.numRegs, fa.localSizeBytes, fa.sharedSizeBytes, fa.maxThreadsPerBlock, block,
            blocksPerSM, active, 100.0 * active / p.maxThreadsPerMultiProcessor, active * p.multiProcessorCount);
}

static void print_info(FILE* f) {
    cudaDeviceProp p;
    CUDA_CHECK(cudaGetDeviceProperties(&p, 0));
    int rt = 0, drv = 0, clock_khz = 0;
    CUDA_CHECK(cudaRuntimeGetVersion(&rt));
    CUDA_CHECK(cudaDriverGetVersion(&drv));
    CUDA_CHECK(cudaDeviceGetAttribute(&clock_khz, cudaDevAttrClockRate, 0));   // works on CUDA 11, 12 and 13
    fprintf(f, "GPU            : %s (compute capability %d.%d)\n", p.name, p.major, p.minor);
    fprintf(f, "SMs            : %d, max %d threads/SM, %d regs/SM, warp size %d\n",
            p.multiProcessorCount, p.maxThreadsPerMultiProcessor, p.regsPerMultiprocessor, p.warpSize);
    fprintf(f, "Clock          : %d MHz, global memory %.0f MiB, L2 %d KiB, constant %zu KiB, shared/block %zu KiB\n",
            clock_khz / 1000, p.totalGlobalMem / 1048576.0, p.l2CacheSize / 1024, p.totalConstMem / 1024,
            p.sharedMemPerBlock / 1024);
    fprintf(f, "CUDA           : runtime %d.%d, driver supports %d.%d\n", rt / 1000, (rt % 1000) / 10, drv / 1000,
            (drv % 1000) / 10);
    fprintf(f, "CPU threads    : %d (OpenMP %s)\n", max_cpu_threads(),
#ifdef _OPENMP
            "enabled"
#else
            "disabled"
#endif
    );
    fprintf(f, "Kernel resources (block = %d):\n", g_block);
    kernel_line(f, "rsa_modexp_kernel<Global>", rsa_modexp_kernel<KeyMem::Global>, g_block);
    kernel_line(f, "rsa_modexp_kernel<Shared>", rsa_modexp_kernel<KeyMem::Shared>, g_block);
    kernel_line(f, "rsa_modexp_kernel<Constant>", rsa_modexp_kernel<KeyMem::Constant>, g_block);
    kernel_line(f, "rsa_crt_kernel", rsa_crt_kernel, g_block);
}

// =============================================================================
//  MODE: demo  - readable end-to-end walk-through
// =============================================================================
static int run_demo(int n) {
    static const char* users[] = {"alice", "bob", "carol", "dave", "erin", "frank", "grace", "heidi",
                                  "ivan", "judy", "mallory", "niaj", "olivia", "peggy", "rupert", "sybil"};
    printf("\n==================== DEMO: %d simultaneous HTTPS connections ====================\n", n);
    printf("Server RSA-%d public key: n = 0x%s...%s (%d bits), e = %u\n", RSA_KEY_BITS,
           bn_hex(RSA_N).substr(0, 16).c_str(), bn_hex(RSA_N).substr(NL * 8 - 8).c_str(),
           bit_length(RSA_N, NL), RSA_E);

    Batch plain(n), cipher(n), recovered(n), recovered_std(n);
    Rng r(2026);
    for (int i = 0; i < n; ++i) {
        char buf[160];
        snprintf(buf, sizeof buf, "conn#%04d user=%s session-key=%08x%08x%08x%08x", i + 1,
                 users[i % 16], r.u32(), r.u32(), r.u32(), r.u32());
        uint32_t x[NL];
        text_to_bn(buf, x);
        plain.set(i, x);
    }

    printf("\n(1) Clients encrypt their session keys with the server's PUBLIC key: c = m^e mod n\n");
    gpu_rsa_batch(Op::Encrypt, KeyMem::Constant, plain, cipher, g_block, true);

    printf("\n(2) Server decrypts ALL connections at once with its PRIVATE key (CRT): m = c^d mod n\n");
    GpuTimes t = gpu_rsa_batch(Op::DecryptCRT, KeyMem::Constant, cipher, recovered, g_block, true);
    printf("      GPU time: H2D %.3f ms | kernel %.3f ms | D2H %.3f ms\n", t.h2d_ms, t.kernel_ms, t.d2h_ms);

    printf("\n(3) Cross-check with plain (non-CRT) GPU decryption  m = c^d mod n\n");
    gpu_rsa_batch(Op::Decrypt, KeyMem::Constant, cipher, recovered_std, g_block);

    printf("\n%-5s | %-26s | %-66s | %s\n", "conn", "ciphertext (first 24 hex)", "decrypted plaintext", "check");
    printf("------+----------------------------+--------------------------------------------------------------------+------\n");
    int ok = 0;
    for (int i = 0; i < n; ++i) {
        uint32_t m[NL], c[NL], d[NL], d2[NL];
        plain.get(i, m); cipher.get(i, c); recovered.get(i, d); recovered_std.get(i, d2);
        const bool good = !memcmp(m, d, sizeof m) && !memcmp(m, d2, sizeof m) && memcmp(m, c, sizeof m);
        ok += good;
        printf("%5d | %s.. | %-66s | %s\n", i + 1, bn_hex(c).substr(0, 24).c_str(), bn_to_text(d).c_str(),
               good ? "OK" : "FAIL");
    }

    // Tamper test: flip ONE bit of one ciphertext
    uint32_t c0[NL], m0[NL], dm[NL];
    cipher.get(0, c0);
    plain.get(0, m0);
    c0[5] ^= 1u << 7;
    rsa_crt_decrypt<NL>(dm, c0, g_crt);
    printf("\nTamper test: flipping 1 bit of ciphertext #1 decrypts to 0x%s...  (%s)\n",
           bn_hex(dm).substr(0, 32).c_str(),
           memcmp(dm, m0, sizeof dm) ? "garbage, as expected" : "UNEXPECTED");
    printf("\nDEMO RESULT: %d / %d connections decrypted correctly\n", ok, n);
    return ok == n ? 0 : 1;
}

// =============================================================================
//  MODE: verify  - correctness test suite
// =============================================================================
static bool report(const char* what, int bad, int total) {
    printf("  %-58s : %s (%d/%d match)\n", what, bad ? "FAIL" : "PASS", total - bad, total);
    return bad == 0;
}

static int run_verify(int n) {
    printf("\n==================== VERIFY: %d messages ====================\n", n);
    Batch M = random_batch(n, 12345);
    // edge cases: 0, 1, 2, n-1
    uint32_t x[NL];
    bn_set_word<NL>(x, 0); M.set(0, x);
    bn_set_word<NL>(x, 1); M.set(1, x);
    bn_set_word<NL>(x, 2); M.set(2, x);
    memcpy(x, RSA_N, sizeof x); x[0] -= 1; M.set(3, x);          // n is odd, so no borrow

    Batch C(n), Ccpu(n), D(n);
    bool all = true;
    const int T = max_cpu_threads();

    gpu_rsa_batch(Op::Encrypt, KeyMem::Constant, M, C, g_block);
    cpu_rsa_batch(Op::Encrypt, M, Ccpu, T);
    all &= report("GPU encryption == CPU encryption (c = m^e mod n)", count_mismatches(C, Ccpu), n);

    for (KeyMem mem : {KeyMem::Global, KeyMem::Shared, KeyMem::Constant}) {
        std::fill(D.w.begin(), D.w.end(), 0u);
        gpu_rsa_batch(Op::Decrypt, mem, C, D, g_block);
        char what[96];
        snprintf(what, sizeof what, "GPU decrypt, key in %s memory == original", mem_name(mem));
        all &= report(what, count_mismatches(D, M), n);
    }
    std::fill(D.w.begin(), D.w.end(), 0u);
    gpu_rsa_batch(Op::DecryptCRT, KeyMem::Constant, C, D, g_block);
    all &= report("GPU CRT decrypt == original", count_mismatches(D, M), n);

    std::fill(D.w.begin(), D.w.end(), 0u);
    cpu_rsa_batch(Op::DecryptCRT, C, D, T);
    all &= report("CPU CRT decrypt == original", count_mismatches(D, M), n);

    // Odd sizes exercise the "if (index < N)" boundary check (partial last block)
    for (int odd : {1, 31, 33, 65, 257, 999}) {
        if (odd > n) continue;
        Batch Ms(odd), Cs(odd), Ds(odd);
        for (int i = 0; i < odd; ++i) { M.get(i, x); Ms.set(i, x); }
        gpu_rsa_batch(Op::Encrypt, KeyMem::Constant, Ms, Cs, g_block);
        gpu_rsa_batch(Op::DecryptCRT, KeyMem::Constant, Cs, Ds, g_block);
        char what[96];
        snprintf(what, sizeof what, "Round trip with batch size %d (partial block)", odd);
        all &= report(what, count_mismatches(Ds, Ms), odd);
    }

    // Dump for the independent Python checker (tools/verify.py)
    std::filesystem::create_directories(g_outdir);
    const std::string path = g_outdir + "/verify_dump.txt";
    FILE* f = fopen(path.c_str(), "w");
    if (!f) { perror(path.c_str()); return 1; }
    fprintf(f, "n %s\ne %x\nd %s\n", bn_hex(RSA_N).c_str(), RSA_E, bn_hex(RSA_D).c_str());
    uint32_t c[NL];
    for (int i = 0; i < n; ++i) {
        M.get(i, x); C.get(i, c);
        fprintf(f, "%s %s\n", bn_hex(x).c_str(), bn_hex(c).c_str());
    }
    fclose(f);
    printf("  wrote %s (checked independently by tools/verify.py)\n", path.c_str());
    printf("VERIFY RESULT: %s\n", all ? "ALL TESTS PASSED" : "SOME TESTS FAILED");
    return all ? 0 : 1;
}

// =============================================================================
//  MODE: bench  - performance experiments
// =============================================================================
static FILE* open_csv(const char* name, const char* header) {
    std::filesystem::create_directories(g_outdir);
    const std::string path = g_outdir + "/" + name;
    FILE* f = fopen(path.c_str(), "w");
    if (!f) { perror(path.c_str()); exit(EXIT_FAILURE); }
    fprintf(f, "%s\n", header);
    return f;
}

static void must_match(const Batch& a, const Batch& b, const char* ctx) {
    const int bad = count_mismatches(a, b);
    if (bad) { fprintf(stderr, "BENCH ABORTED: %d wrong results in %s\n", bad, ctx); exit(EXIT_FAILURE); }
}

static int run_bench() {
    printf("\n==================== BENCHMARKS ====================\n");
    const int T = max_cpu_threads();
    const int REPS = 3;
    gpu_warmup();

    {   // kernel resource usage
        std::filesystem::create_directories(g_outdir);
        FILE* f = fopen((g_outdir + "/device_info.txt").c_str(), "w");
        print_info(stdout);
        if (f) { print_info(f); fclose(f); }
    }

    // ---------------- Experiment 1: scaling with number of connections ------
    printf("\n[E1] Decryption time vs. number of simultaneous connections (block=%d)\n", g_block);
    printf("%6s | %11s %11s | %11s %11s | %11s %11s | %11s %11s\n", "conns", "GPU std", "GPU CRT",
           "CPU1 std", "CPU1 CRT", "CPU8 std", "CPU8 CRT", "GPUcrt e2e", "GPUstd e2e");
    FILE* f1 = open_csv("e1_scaling.csv",
                        "connections,gpu_std_kernel_ms,gpu_std_e2e_ms,gpu_crt_kernel_ms,gpu_crt_e2e_ms,"
                        "cpu1_std_ms,cpu1_crt_ms,cpuN_std_ms,cpuN_crt_ms,cpu_threads");
    const std::vector<int> sizes = {1, 8, 32, 64, 128, 256, 384, 512, 768, 1024, 1152, 1536, 2048};
    GpuTimes last_std, last_crt;
    for (int n : sizes) {
        Batch M = random_batch(n, 1000 + n), C(n), D(n);
        cpu_rsa_batch(Op::Encrypt, M, C, T);
        GpuTimes gs = gpu_median(Op::Decrypt, KeyMem::Constant, C, D, g_block, REPS);   must_match(D, M, "gpu std");
        GpuTimes gc = gpu_median(Op::DecryptCRT, KeyMem::Constant, C, D, g_block, REPS); must_match(D, M, "gpu crt");
        const double c1s = cpu_rsa_batch(Op::Decrypt, C, D, 1);     must_match(D, M, "cpu1 std");
        const double c1c = cpu_rsa_batch(Op::DecryptCRT, C, D, 1);  must_match(D, M, "cpu1 crt");
        const double cNs = cpu_rsa_batch(Op::Decrypt, C, D, T);     must_match(D, M, "cpuN std");
        const double cNc = cpu_rsa_batch(Op::DecryptCRT, C, D, T);  must_match(D, M, "cpuN crt");
        printf("%6d | %9.2fms %9.2fms | %9.1fms %9.1fms | %9.1fms %9.1fms | %9.2fms %9.2fms\n", n,
               gs.kernel_ms, gc.kernel_ms, c1s, c1c, cNs, cNc, gc.e2e(), gs.e2e());
        fprintf(f1, "%d,%.4f,%.4f,%.4f,%.4f,%.4f,%.4f,%.4f,%.4f,%d\n", n, gs.kernel_ms, gs.e2e(), gc.kernel_ms,
                gc.e2e(), c1s, c1c, cNs, cNc, T);
        fflush(f1);
        last_std = gs; last_crt = gc;
    }
    fclose(f1);

    // ---------------- Experiment 2: time breakdown of the 5-step pipeline ---
    const int NB = sizes.back();
    FILE* f2 = open_csv("e2_breakdown.csv", "variant,connections,alloc_ms,h2d_ms,kernel_ms,d2h_ms,free_ms");
    fprintf(f2, "standard,%d,%.4f,%.4f,%.4f,%.4f,%.4f\n", NB, last_std.alloc_ms, last_std.h2d_ms,
            last_std.kernel_ms, last_std.d2h_ms, last_std.free_ms);
    fprintf(f2, "crt,%d,%.4f,%.4f,%.4f,%.4f,%.4f\n", NB, last_crt.alloc_ms, last_crt.h2d_ms, last_crt.kernel_ms,
            last_crt.d2h_ms, last_crt.free_ms);
    fclose(f2);
    printf("\n[E2] Pipeline breakdown @ %d connections (CRT): alloc %.3f | H2D %.3f | kernel %.2f | D2H %.3f | "
           "free %.3f ms\n", NB, last_crt.alloc_ms, last_crt.h2d_ms, last_crt.kernel_ms, last_crt.d2h_ms,
           last_crt.free_ms);

    // ---------------- Experiment 3: memory hierarchy for the key ------------
    const int NM = 1024;
    printf("\n[E3] Where should the key live?  (standard decryption, %d connections, block=%d)\n", NM, g_block);
    FILE* f3 = open_csv("e3_memory.csv", "key_memory,connections,kernel_ms,throughput_ops_per_s");
    {
        Batch M = random_batch(NM, 77), C(NM), D(NM);
        cpu_rsa_batch(Op::Encrypt, M, C, T);
        for (KeyMem mem : {KeyMem::Global, KeyMem::Shared, KeyMem::Constant}) {
            GpuTimes g = gpu_median(Op::Decrypt, mem, C, D, g_block, REPS);
            must_match(D, M, mem_name(mem));
            printf("  key in %-8s memory : kernel %9.2f ms  -> %8.1f decryptions/s\n", mem_name(mem), g.kernel_ms,
                   NM / (g.kernel_ms / 1000.0));
            fprintf(f3, "%s,%d,%.4f,%.2f\n", mem_name(mem), NM, g.kernel_ms, NM / (g.kernel_ms / 1000.0));
        }
    }
    fclose(f3);

    // ---------------- Experiment 4: block size ------------------------------
    printf("\n[E4] Effect of threads per block (%d connections)\n", NM);
    FILE* f4 = open_csv("e4_blocksize.csv", "block,connections,std_kernel_ms,crt_kernel_ms");
    {
        Batch M = random_batch(NM, 99), C(NM), D(NM);
        cpu_rsa_batch(Op::Encrypt, M, C, T);
        cudaFuncAttributes fa_s, fa_c;
        CUDA_CHECK(cudaFuncGetAttributes(&fa_s, rsa_modexp_kernel<KeyMem::Constant>));
        CUDA_CHECK(cudaFuncGetAttributes(&fa_c, rsa_crt_kernel));
        for (int b : {32, 64, 128, 256, 512}) {
            double s = -1, c = -1;
            if (b <= fa_s.maxThreadsPerBlock) {
                s = gpu_median(Op::Decrypt, KeyMem::Constant, C, D, b, REPS).kernel_ms;
                must_match(D, M, "block std");
            }
            if (b <= fa_c.maxThreadsPerBlock) {
                c = gpu_median(Op::DecryptCRT, KeyMem::Constant, C, D, b, REPS).kernel_ms;
                must_match(D, M, "block crt");
            }
            printf("  block %4d : standard %9.2f ms | CRT %9.2f ms%s\n", b, s, c,
                   (s < 0 || c < 0) ? "   (-1 = block too large for this kernel's register usage)" : "");
            fprintf(f4, "%d,%d,%.4f,%.4f\n", b, NM, s, c);
        }
    }
    fclose(f4);

    // ---------------- Experiment 5: encryption (public key, e = 65537) ------
    const int NE = 4096;
    printf("\n[E5] Encryption with e = 65537 (%d connections)\n", NE);
    FILE* f5 = open_csv("e5_encrypt.csv", "connections,gpu_kernel_ms,gpu_e2e_ms,cpu1_ms,cpuN_ms,cpu_threads");
    {
        Batch M = random_batch(NE, 5), C(NE), Cc(NE);
        GpuTimes g = gpu_median(Op::Encrypt, KeyMem::Constant, M, C, g_block, REPS);
        const double c1 = cpu_rsa_batch(Op::Encrypt, M, Cc, 1);
        must_match(C, Cc, "encrypt gpu vs cpu1");
        const double cN = cpu_rsa_batch(Op::Encrypt, M, Cc, T);
        must_match(C, Cc, "encrypt gpu vs cpuN");
        printf("  GPU kernel %.2f ms (e2e %.2f ms) | CPU 1 thread %.1f ms | CPU %d threads %.1f ms\n", g.kernel_ms,
               g.e2e(), c1, T, cN);
        fprintf(f5, "%d,%.4f,%.4f,%.4f,%.4f,%d\n", NE, g.kernel_ms, g.e2e(), c1, cN, T);
    }
    fclose(f5);
    printf("\nBENCH COMPLETE - CSV files written to %s/\n", g_outdir.c_str());
    return 0;
}

// =============================================================================
int main(int argc, char** argv) {
    setvbuf(stdout, nullptr, _IOLBF, 0);   // line-buffered log output
    std::string mode = "all";
    int n = -1;
    for (int i = 1; i < argc; ++i) {
        const std::string a = argv[i];
        if (a == "--out" && i + 1 < argc) g_outdir = argv[++i];
        else if (a == "--chunk" && i + 1 < argc) g_chunk = std::max(1, atoi(argv[++i]));
        else if (a == "--block" && i + 1 < argc) g_block = std::max(32, atoi(argv[++i]));
        else if (a == "demo" || a == "verify" || a == "bench" || a == "info" || a == "all") mode = a;
        else if (atoi(a.c_str()) > 0) n = atoi(a.c_str());
        else { fprintf(stderr, "usage: %s [demo|verify|bench|info|all] [N] [--block B] [--chunk C] [--out DIR]\n", argv[0]); return 2; }
    }

    int devices = 0;
    CUDA_CHECK(cudaGetDeviceCount(&devices));
    if (devices == 0) { fprintf(stderr, "No CUDA device found\n"); return 1; }
    build_keys();

    int rc = 0;
    if (mode == "info")   { print_info(stdout); return 0; }
    if (mode == "demo"   || mode == "all") rc |= run_demo(n > 0 ? n : 8);
    if (mode == "verify" || mode == "all") rc |= run_verify(n > 0 ? n : 1000);
    if (mode == "bench"  || mode == "all") rc |= run_bench();
    return rc;
}


In [ ]:
%%writefile Makefile
# Makefile - CUDA RSA case study
#   make                 build ./rsa_cuda  (default GPU arch: sm_61 = GeForce MX250 / Pascal)
#   make ARCH=sm_75      e.g. Tesla T4 (Google Colab), sm_86 = RTX 30xx, sm_89 = RTX 40xx
#   make run             demo + verification + benchmarks
#   make regs            show per-kernel register / spill usage reported by ptxas

NVCC     ?= nvcc
ARCH     ?= sm_61
CCBIN    ?=
NVFLAGS  := -O3 -std=c++17 -arch=$(ARCH) -lineinfo
HOSTOPT  := -Xcompiler -O3,-march=native,-fopenmp
LDFLAGS  := -lgomp
ifneq ($(CCBIN),)
  NVFLAGS += -ccbin $(CCBIN)
endif

SRC := src/rsa_cuda.cu
DEP := src/bigint.cuh src/rsa_key.h

# The fully unrolled 64-limb carry chains are deep enough to overflow the
# default 8 MB stack of NVIDIA's front end (cicc -> "Segmentation fault"),
# so the stack limit is raised for the compiler process first.
STACK := ulimit -s unlimited 2>/dev/null || ulimit -s $$(ulimit -H -s);

rsa_cuda: $(SRC) $(DEP)
	$(STACK) $(NVCC) $(NVFLAGS) $(HOSTOPT) $(SRC) -o $@ $(LDFLAGS) $(EXTRA_LDFLAGS)

src/rsa_key.h:
	python3 tools/keygen.py

regs: $(SRC) $(DEP)
	$(STACK) $(NVCC) $(NVFLAGS) -Xptxas -v -c $(SRC) -o /dev/null 2>&1 | grep -E "Compiling|registers|spill"

run: rsa_cuda
	./rsa_cuda all

clean:
	rm -f rsa_cuda

.PHONY: regs run clean


In [ ]:
%%writefile tools/verify.py
#!/usr/bin/env python3
"""
verify.py - Independent correctness check of the CUDA results.

Reads results/verify_dump.txt written by `rsa_cuda verify` (the key and every
(plaintext m, GPU ciphertext c) pair) and re-computes both directions with
Python's built-in arbitrary-precision pow(), which shares no code with the
CUDA/C++ implementation:
    pow(m, e, n) == c      (GPU encryption is correct)
    pow(c, d, n) == m      (the ciphertext really decrypts to the plaintext)
"""
import os
import sys
import time

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
path = sys.argv[1] if len(sys.argv) > 1 else os.path.join(ROOT, "results", "verify_dump.txt")

with open(path) as f:
    lines = [ln.split() for ln in f if ln.strip()]

hdr = {k: v for k, v in lines[:3]}
n, e, d = int(hdr["n"], 16), int(hdr["e"], 16), int(hdr["d"], 16)
pairs = [(int(a, 16), int(b, 16)) for a, b in lines[3:]]

t0 = time.perf_counter()
enc_bad = sum(pow(m, e, n) != c for m, c in pairs)
dec_bad = sum(pow(c, d, n) != m for m, c in pairs)
dt = time.perf_counter() - t0

print(f"Independent Python check of {len(pairs)} GPU results (RSA-{n.bit_length()}):")
print(f"  pow(m, e, n) == c_gpu : {len(pairs) - enc_bad}/{len(pairs)} {'PASS' if enc_bad == 0 else 'FAIL'}")
print(f"  pow(c, d, n) == m     : {len(pairs) - dec_bad}/{len(pairs)} {'PASS' if dec_bad == 0 else 'FAIL'}")
print(f"  (Python single-thread time for {len(pairs)} decryptions: {dt:.2f} s)")
sys.exit(0 if enc_bad == dec_bad == 0 else 1)


In [ ]:
%%writefile tools/plot.py
#!/usr/bin/env python3
"""
plot.py - Turns the benchmark CSV files in results/ into report figures (PNG).
Requires matplotlib.
"""
import csv
import os

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
RES = os.path.join(ROOT, "results")
FIG = os.path.join(RES, "figures")
os.makedirs(FIG, exist_ok=True)

# One colour per series, used consistently in every figure
C = {
    "gpu_crt": "#1f6f3f", "gpu_std": "#6fbf73",
    "cpuN_crt": "#1f4e9c", "cpuN_std": "#7aa6e0",
    "cpu1_crt": "#b3541e", "cpu1_std": "#f0a868",
}
plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 200, "font.size": 9.5, "axes.titlesize": 10.5,
    "axes.titleweight": "bold", "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "legend.frameon": False,
})


def read(name):
    with open(os.path.join(RES, name)) as f:
        return list(csv.DictReader(f))


def col(rows, key, typ=float):
    return [typ(r[key]) for r in rows]


def save(fig, name):
    fig.tight_layout()
    fig.savefig(os.path.join(FIG, name), bbox_inches="tight")
    plt.close(fig)
    print("  wrote results/figures/" + name)


e1 = read("e1_scaling.csv")
N = col(e1, "connections", int)
T = int(e1[0]["cpu_threads"])
series = [
    ("gpu_crt", "gpu_crt_kernel_ms", "GPU  CRT"),
    ("gpu_std", "gpu_std_kernel_ms", "GPU  standard"),
    ("cpuN_crt", "cpuN_crt_ms", f"CPU {T} threads  CRT"),
    ("cpuN_std", "cpuN_std_ms", f"CPU {T} threads  standard"),
    ("cpu1_crt", "cpu1_crt_ms", "CPU 1 thread  CRT"),
    ("cpu1_std", "cpu1_std_ms", "CPU 1 thread  standard"),
]

# ---- Fig 1: time vs connections -------------------------------------------
fig, ax = plt.subplots(figsize=(6.4, 4.0))
for key, colname, label in series:
    ax.plot(N, col(e1, colname), marker="o", ms=3.5, lw=1.8, color=C[key], label=label,
            ls="-" if "crt" in key else "--")
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xlabel("Simultaneous connections (RSA-2048 decryptions)")
ax.set_ylabel("Time (ms, log scale)")
ax.set_title("Fig. 1  Decryption time vs. batch size")
ax.legend(fontsize=8, ncol=2)
save(fig, "fig1_time_vs_connections.png")

# ---- Fig 2: throughput -----------------------------------------------------
fig, ax = plt.subplots(figsize=(6.4, 4.0))
for key, colname, label in series:
    thr = [n / (t / 1000.0) for n, t in zip(N, col(e1, colname))]
    ax.plot(N, thr, marker="o", ms=3.5, lw=1.8, color=C[key], label=label, ls="-" if "crt" in key else "--")
ax.set_xscale("log", base=2)
ax.set_xlabel("Simultaneous connections")
ax.set_ylabel("Throughput (decryptions / second)")
ax.set_title("Fig. 2  Throughput: the GPU needs many connections to fill up")
ax.legend(fontsize=8, ncol=2)
save(fig, "fig2_throughput.png")

# ---- Fig 3: speed-up of GPU (CRT) over CPU ---------------------------------
fig, ax = plt.subplots(figsize=(6.4, 3.8))
g = col(e1, "gpu_crt_kernel_ms")
ge = col(e1, "gpu_crt_e2e_ms")
c1 = col(e1, "cpu1_crt_ms")
cN = col(e1, "cpuN_crt_ms")
ax.plot(N, [a / b for a, b in zip(c1, g)], marker="o", ms=3.5, lw=1.8, color=C["cpu1_crt"],
        label="vs CPU 1 thread (kernel time)")
ax.plot(N, [a / b for a, b in zip(c1, ge)], marker="s", ms=3, lw=1.2, color=C["cpu1_crt"], ls=":",
        label="vs CPU 1 thread (incl. PCIe copies)")
ax.plot(N, [a / b for a, b in zip(cN, g)], marker="o", ms=3.5, lw=1.8, color=C["cpuN_crt"],
        label=f"vs CPU {T} threads (kernel time)")
ax.axhline(1.0, color="#555", lw=0.9)
ax.text(N[0], 1.04, "break-even", fontsize=7.5, color="#555")
ax.set_xscale("log", base=2)
ax.set_xlabel("Simultaneous connections")
ax.set_ylabel("Speed-up (x)")
ax.set_title("Fig. 3  GPU speed-up for CRT decryption")
ax.legend(fontsize=8)
save(fig, "fig3_speedup.png")

# ---- Fig 4: pipeline breakdown ----------------------------------------------
e2 = read("e2_breakdown.csv")
fig, ax = plt.subplots(figsize=(6.4, 2.6))
steps = [("alloc_ms", "1 cudaMalloc", "#999999"), ("h2d_ms", "2 Memcpy H->D", "#e0a030"),
         ("kernel_ms", "3 Kernel", "#1f6f3f"), ("d2h_ms", "4 Memcpy D->H", "#c05050"),
         ("free_ms", "5 cudaFree", "#666666")]
labels = []
for i, r in enumerate(e2):
    left = 0.0
    total = sum(float(r[k]) for k, _, _ in steps)
    for k, lab, colr in steps:
        v = float(r[k])
        ax.barh(i, v / total * 100, left=left, color=colr, label=lab if i == 0 else None, height=0.55)
        left += v / total * 100
    labels.append(f"{'CRT' if r['variant'] == 'crt' else 'Standard'}\n({r['connections']} conns)")
    ax.text(101, i, f"kernel = {float(r['kernel_ms']) / total * 100:.2f}% of {total:.1f} ms",
            va="center", fontsize=8)
ax.set_yticks(range(len(e2)))
ax.set_yticklabels(labels)
ax.set_xlim(0, 135)
ax.set_xlabel("Share of total GPU pipeline time (%)")
ax.set_title("Fig. 4  Where the time goes in the 5-step CUDA pipeline")
ax.legend(fontsize=7.5, ncol=5, loc="upper center", bbox_to_anchor=(0.42, -0.35))
ax.grid(axis="y", visible=False)
save(fig, "fig4_breakdown.png")

# ---- Fig 5: memory hierarchy ------------------------------------------------
e3 = read("e3_memory.csv")
fig, ax = plt.subplots(figsize=(5.2, 3.2))
names = [r["key_memory"] for r in e3]
vals = col(e3, "kernel_ms")
bars = ax.bar(names, vals, color=["#c05050", "#e0a030", "#1f6f3f"], width=0.55)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.0f} ms", ha="center", va="bottom", fontsize=8.5)
ax.set_ylabel("Kernel time (ms)")
ax.set_xlabel("Memory space holding the RSA key (n, R^2, d)")
ax.set_title(f"Fig. 5  Key placement ({e3[0]['connections']} decryptions)")
ax.set_ylim(0, max(vals) * 1.15)
save(fig, "fig5_memory.png")

# ---- Fig 6: block size ------------------------------------------------------
e4 = read("e4_blocksize.csv")
fig, ax = plt.subplots(figsize=(5.8, 3.2))
blocks = [r["block"] for r in e4]
x = range(len(blocks))
s = col(e4, "std_kernel_ms")
c = col(e4, "crt_kernel_ms")
w = 0.38
ax.bar([i - w / 2 for i in x], [v if v > 0 else 0 for v in s], w, color=C["gpu_std"], label="standard")
ax.bar([i + w / 2 for i in x], [v if v > 0 else 0 for v in c], w, color=C["gpu_crt"], label="CRT")
for i, (vs, vc) in enumerate(zip(s, c)):
    if vs < 0:
        ax.text(i - w / 2, 5, "n/a", ha="center", fontsize=7, rotation=90)
    if vc < 0:
        ax.text(i + w / 2, 5, "n/a", ha="center", fontsize=7, rotation=90)
ax.set_xticks(list(x))
ax.set_xticklabels(blocks)
ax.set_xlabel("Threads per block")
ax.set_ylabel("Kernel time (ms)")
ax.set_title(f"Fig. 6  Effect of block size ({e4[0]['connections']} decryptions)")
ax.legend(fontsize=8)
save(fig, "fig6_blocksize.png")
print("all figures written")


## Build
The first compile takes about a minute (the 2048-bit arithmetic is fully unrolled).

In [ ]:
!make ARCH={ARCH} 2>&1 | grep -v '^ulimit' | tail -5
!ls -la rsa_cuda

## Demo: 8 simultaneous HTTPS connections
Clients encrypt session keys with the server's public key; the server decrypts all of them at once on the GPU, one thread per connection. Watch for the `[Step 1]` ... `[Step 5]` lines.

In [ ]:
!./rsa_cuda info
!./rsa_cuda demo 8

## Correctness suite + independent check
GPU vs CPU vs original message on 1,000 messages, the three key-memory variants, CRT, and odd batch sizes (boundary check). Then Python's own `pow()` re-checks every GPU result.

In [ ]:
!./rsa_cuda verify 1000
!python3 tools/verify.py

## Optional: full benchmark (about 10 minutes on Colab)
Uncomment and run. Colab's CPU has only 2 cores, so the CPU baseline is slower than on a laptop.

In [ ]:
# !./rsa_cuda bench
# !python3 tools/plot.py
# from IPython.display import Image, display
# for f in sorted(os.listdir('results/figures')): display(Image('results/figures/' + f))